https://arxiv.org/abs/1608.06993

In [1]:
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary
import numpy as np
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from torch.optim import Adam
from sklearn.metrics import accuracy_score
import math

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available else torch.device('cpu')

In [ ]:
class DenseLayer(nn.Module):
    def __init__(self, num_input_features, growth_rate):
        super().__init__()
        self.batchnorm1 = nn.BatchNorm2d(num_features=num_input_features)
        self.conv1x1 = nn.Conv2d(in_channels=num_input_features, out_channels=4*growth_rate, kernel_size=(1,1))
        self.batchnorm2 = nn.BatchNorm2d(num_features=4*growth_rate)
        self.conv3x3 = nn.Conv2d(in_channels=4*growth_rate, out_channels=growth_rate, kernel_size=(3,3), padding=1)

    def forward(self, previous_feature_maps):
        x = self.batchnorm1(previous_feature_maps)
        x = F.relu(x)
        x = self.conv1x1(x)
        x = self.batchnorm2(x)
        x = F.relu(x)
        x = self.conv3x3(x)
        return torch.cat((previous_feature_maps, x), dim=1)
        

In [ ]:
class DenseBlock(nn.Module):
    def __init__(self, num_layers, num_input_features, growth_rate):
        super().__init__()
        input_features = num_input_features
        self.dense_layers = nn.ModuleList()
        for _ in range(num_layers):
            self.dense_layers.append(DenseLayer(num_input_features=input_features, growth_rate=growth_rate))
            input_features += growth_rate
    
    def forward(self, x):
        for layer in self.dense_layers:
            x = layer(x)
        
        return x

In [ ]:
class TransitionLayer(nn.Module):
    def __init__(self, num_input_features, compression=0.5):
        super().__init__()
        self.batchnorm = nn.BatchNorm2d(num_features=num_input_features)
        self.conv1x1 = nn.Conv2d(in_channels=num_input_features, out_channels=math.floor(num_input_features * compression), kernel_size=(1,1))
        self.avgpool = nn.AvgPool2d(kernel_size=(2,2), stride=2)
    def forward(self, x):
        x = self.batchnorm(x)
        x = F.relu(x)
        x = self.conv1x1(x)
        x = self.avgpool(x)
        return x

In [ ]:
class DenseNet121(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=(7,7), stride=2, padding=3)
        self.bn1 = nn.BatchNorm2d(num_features=64)
        self.maxpool = nn.MaxPool2d(kernel_size=(3,3), stride=2, padding=1)
        self.dense_blocks = nn.ModuleList()

        dense_block_layers = [6, 12, 24, 16]
        channel_count = 64
        for idx in range(4):
            block = DenseBlock(num_layers=dense_block_layers[idx], num_input_features=channel_count, growth_rate=32)
            channel_count = channel_count + 32 * dense_block_layers[idx]
            transition = TransitionLayer(num_input_features=channel_count, compression=0.5)
            channel_count =  channel_count // 2
            self.dense_blocks.append(block)

            if idx != 3:
                self.dense_blocks.append(transition)

        self.bn2 = nn.BatchNorm2d(channel_count)
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(channel_count, num_classes)
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.maxpool(x)

        for block in self.dense_blocks:
            x = block(x)

        x = self.bn2(x)
        x = F.relu(x)
        x = self.avgpool(x)
        x = torch.flatten(x)
        x = self.fc(x)
        return x